📅 **论文年份 (Year):2015 年**  
*Pointer Networks — Vinyals, Fortunato, Jaitly*

# Paper 6: Pointer Networks(指针网络)
## Oriol Vinyals, Meire Fortunato, Navdeep Jaitly

### Implementation: Attention-based Pointing Mechanism(实现:基于注意力的指向机制)

Pointer Networks use attention to point to input elements, solving combinatorial problems like convex hull and TSP.

指针网络(Pointer Networks)利用注意力(attention)机制来指向输入元素,从而解决凸包(convex hull)和旅行商问题(TSP)等组合优化问题。

## 📖 论文导读

**🎯 这篇文章想解决什么问题(目的):** 传统的序列到序列(Seq2Seq)模型有个"死穴":输出只能从一个固定的词表里挑。但有一类问题的答案本身就是输入的一部分——比如给你 10 个城市规划旅行路线,答案就是这 10 个城市的某种排列;输入换成 20 个城市,"词表"也得跟着变。固定词表的模型对此无能为力。这篇论文就是要让神经网络学会处理"输出是从输入中选出来的"这类问题。

**💡 主要贡献:** 提出了指针网络(Pointer Network),核心想法非常巧妙:不再让模型从词表里"说"出答案,而是让它伸出"手指"直接指向输入中的某个元素。具体做法是把注意力机制(attention)从"辅助工具"变成"输出机制"本身——注意力分数不再用来加权求和,而是直接当作"指向每个输入元素的概率"。这样输出空间自动随输入长度伸缩,再也不需要固定词表。

**🔧 方法:** 模型仍是编码器-解码器结构:编码器先把每个输入元素(比如平面上的点)读一遍并记住;解码器每一步计算当前状态与每个输入元素的"匹配分数",用 softmax 变成概率分布,概率最高的那个元素就是当前要"指"的输出。论文在凸包、旅行商问题(TSP)、三角剖分等经典组合优化问题上验证了效果——这些问题的共同点是:答案都是输入点的一个排列或子集。

**🌟 意义:** 这篇论文第一次让神经网络能"端到端"地学习组合优化问题,开创了"神经网络做组合优化"这一研究方向。更深远的影响是"指向输入"这一机制本身:后来文本摘要中的复制机制(copy mechanism,直接从原文抄词)、阅读理解中定位答案位置等,都直接借鉴了这个思想。它也是理解"注意力可以有多种用法"的绝佳例子——同一个机制,换个用途就打开了一片新天地。

## 🎯 核心结论 (Key Takeaways)

- **论文解决的死穴:** 传统 Seq2Seq 只能从固定词表里挑输出,可凸包、TSP、排序这类问题的答案本身就是输入的一部分,输入有几个元素"词表"就得有多大。指针网络把注意力分数直接当作"指向每个输入位置的概率",输出空间自动随输入长度伸缩,彻底绕开了固定词表的限制。
- **机制验证:** 本 notebook 用纯 NumPy 实现了指向式注意力(`e_i = v^T·tanh(W1·编码器状态 + W2·解码器状态)`),在 5 个位置的测试中 softmax 后的指针概率之和恰为 1.0000——说明"指针"就是一个合法的、覆盖全部输入位置的概率分布。
- **凸包实验(未训练):** 在 8 个随机点的凸包任务上,未训练的网络给出的指向顺序与真实凸包顺序不一致(基本是乱指的),但每一步的输出都必然落在输入的 8 个位置之内——展示的是"结构"而非"能力":指针机制天生保证输出合法,准确性要靠训练获得。
- **排序任务同理:** 数字排序被改写成"按从小到大依次指向哪些输入位置",答案就是 `np.argsort` 给出的索引排列——它和凸包本质相同:输出都是输入位置的一个排列,这正是指针网络最擅长的题型。
- **可视化亮点:** 逐步画出的注意力图(红圈大小 = 指向概率)让"网络此刻在指谁"一目了然,这也是理解 copy 机制、阅读理解答案定位等后续工作的直观入口。
- **一句话带走:** 把注意力从"加权求和的辅助工具"改成"直接选择输入的输出机制",神经网络就能端到端地做组合优化——同一个机制,换个用法就是一篇开创性论文。


## 🤯 反常识的发现 (Counterintuitive Findings)

- **常识认为:神经网络的输出层大小是造网络时就焊死的,训练时见过多长的输入,就只能处理多长的输入。** 但这篇论文发现,指针网络在只见过 50 个点的凸包训练后,直接拿 500 个点测试照样能工作——因为它的输出不是从固定词表里"说"出来的,而是"指回输入":输入有几个元素,输出空间就自动有几个选项,长度泛化是结构自带的,不是学出来的。

- **常识认为:注意力只是个"辅助工具",算完分数还得拿去加权平均,真正的输出另有输出层负责。** 但这篇论文把注意力的最后一步(加权求和)直接砍掉了——softmax 后的注意力分数本身就是输出:概率最大的那个输入位置就是答案。本 notebook 验证了这一点:5 个位置的指针概率之和恰为 1.0000,注意力分布天生就是一个合法的"选谁"概率分布。

- **常识认为:一个完全没训练的网络,输出应该是彻底无意义的垃圾。** 但 notebook 的凸包实验(8 个随机点)显示,未训练的指针网络虽然指得乱七八糟,可每一步的输出都必然落在输入的 8 个位置之内——永远不会"指到不存在的东西"。输出的合法性由架构保证,训练只需要负责准确性,这正是"结构先于能力"的一个直观例子。

- **常识认为:凸包、TSP 这类组合优化问题得靠人类精心设计的几何/搜索算法,神经网络这种"函数拟合器"插不上手。** 但论文证明,只给"点集 → 正确顺序"的例子对,纯数据驱动的 seq2seq 就能近似学出这些算法(TSP 上给出接近最优的解)。notebook 里的排序任务也是同一件事:答案不过是 `np.argsort` 的索引排列,网络学的是"指向的顺序",而不是任何显式的排序规则。


#### 💻 代码解读

**做什么:** 导入本笔记本需要的工具库,并固定随机种子,保证每次运行结果一致。

**怎么做:**
- 导入 `numpy`(数值计算,相当于"计算器")和 `matplotlib.pyplot`(画图工具)。
- 从 `scipy.spatial` 导入 `ConvexHull`,后面用它来计算一组点的"凸包"(把所有点包起来的最小凸多边形),作为标准答案。
- 调用 `np.random.seed(42)` 固定随机种子,就像掷骰子前先"锁定"骰子,让每次运行生成的随机数都一样,方便复现实验。

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.spatial import ConvexHull

np.random.seed(42)

## Attention Mechanism for Pointing(用于指向的注意力机制)

#### 💻 代码解读

**做什么:** 实现指针网络的核心——"指向式"注意力机制:给定编码器的各个输入状态和解码器当前状态,算出一个概率分布,表示"下一步应该指向哪个输入元素"。

**怎么做:**
- 先定义 `softmax` 函数:把一组分数变成加起来等于 1 的概率(先减去最大值防止数值溢出,是"稳定版"写法)。
- 定义 `PointerAttention` 类,初始化三组可学习参数 `W1`、`W2`、`v`,相当于三把"打分尺子"。
- `forward` 方法对每个输入位置计算注意力分数:`e_i = v^T * tanh(W1*编码器状态 + W2*解码器状态)`,相当于把"输入本身的信息"和"当前想找什么"揉在一起打分。
- 把所有分数经过 `softmax` 转成概率 `probs`——这就是"指针":概率最大的位置就是最想指向的输入。
- 最后用随机生成的 5 个编码器状态测试一遍,打印概率之和(应为 1.0000)和形状,验证机制正常工作。

In [ ]:
def softmax(x, axis=-1):
    """Stable softmax"""
    # 数值稳定技巧:先减去最大值再取exp,防止exp(大数)溢出;keepdims=True保留维度以便广播
    x_max = np.max(x, axis=axis, keepdims=True)
    exp_x = np.exp(x - x_max)
    return exp_x / np.sum(exp_x, axis=axis, keepdims=True)

class PointerAttention:
    def __init__(self, hidden_size):
        self.hidden_size = hidden_size
        
        # Attention parameters
        # W1作用于编码器状态,W2作用于解码器状态,v把tanh后的向量压成标量分数(加性注意力)
        self.W1 = np.random.randn(hidden_size, hidden_size) * 0.1
        self.W2 = np.random.randn(hidden_size, hidden_size) * 0.1
        self.v = np.random.randn(hidden_size, 1) * 0.1
    
    def forward(self, encoder_states, decoder_state):
        """
        Compute attention scores over input elements
        
        encoder_states: (seq_len, hidden_size) - encoded input
        decoder_state: (hidden_size, 1) - current decoder state
        
        Returns:
        probs: (seq_len, 1) - pointer distribution over inputs
        """
        seq_len = encoder_states.shape[0]
        
        # Compute attention scores
        scores = []
        for i in range(seq_len):
            # e_i = v^T * tanh(W1*encoder_state + W2*decoder_state)
            # 论文核心公式:对每个输入位置i算一个"指向分数"e_i
            # encoder_states[i:i+1]用切片保持二维,形状(1,hidden),转置后为(hidden,1)
            encoder_proj = np.dot(self.W1, encoder_states[i:i+1].T)
            decoder_proj = np.dot(self.W2, decoder_state)
            score = np.dot(self.v.T, np.tanh(encoder_proj + decoder_proj))
            scores.append(score[0, 0])
        
        scores = np.array(scores).reshape(-1, 1)
        
        # Softmax to get probabilities
        # 关键区别:普通注意力用softmax结果加权求和,指针网络直接把它当作"选哪个输入"的分布
        # 输出词表大小=输入序列长度,天然支持可变长度输入
        probs = softmax(scores, axis=0)
        
        return probs, scores

# Test attention
hidden_size = 32
attention = PointerAttention(hidden_size)

# Dummy encoder states and decoder state
seq_len = 5
encoder_states = np.random.randn(seq_len, hidden_size)
decoder_state = np.random.randn(hidden_size, 1)

probs, scores = attention.forward(encoder_states, decoder_state)
print(f"Pointer Network Attention initialized")
print(f"Attention probabilities sum: {probs.sum():.4f}")
print(f"Probabilities shape: {probs.shape}")

## Complete Pointer Network Architecture(完整的指针网络架构)

#### 💻 代码解读

**做什么:** 搭建完整的指针网络(`PointerNetwork`)架构:编码器读入序列,解码器每一步不"生成"新词,而是"用手指"指向输入中的某个元素。

**怎么做:**
- `__init__` 里初始化三部分:编码器 RNN 的权重(`encoder_Wx`、`encoder_Wh`)、解码器 RNN 的权重(`decoder_Wx`、`decoder_Wh`),以及上一格定义的 `PointerAttention` 指针注意力。
- `encode` 方法像"传送带"一样逐个读入输入向量,用 `tanh` 更新隐藏状态 `h`,并把每一步的状态存进 `encoder_states`(相当于给每个输入元素建一张"档案卡")。
- `decode_step` 是解码器走一步:先更新自己的隐藏状态,再调用注意力算出对所有输入位置的指向概率 `probs`。
- `forward` 是完整流程:先编码,再用输入的平均值当"起始信号";每一步用 `np.argmax(probs)` 选出概率最大的位置 `ptr_idx` 作为输出,并把被指向的那个输入元素作为下一步的输入,循环直到输出和输入一样长。
- 关键点:输出的"词表"就是输入的位置,输入有几个元素就有几个选项,完美解决了输出词表随输入长度变化的问题。

In [ ]:
class PointerNetwork:
    def __init__(self, input_size, hidden_size):
        self.input_size = input_size
        self.hidden_size = hidden_size
        
        # Encoder (simple RNN)
        self.encoder_Wx = np.random.randn(hidden_size, input_size) * 0.1
        self.encoder_Wh = np.random.randn(hidden_size, hidden_size) * 0.1
        self.encoder_b = np.zeros((hidden_size, 1))
        
        # Decoder (RNN)
        self.decoder_Wx = np.random.randn(hidden_size, input_size) * 0.1
        self.decoder_Wh = np.random.randn(hidden_size, hidden_size) * 0.1
        self.decoder_b = np.zeros((hidden_size, 1))
        
        # Pointer mechanism
        self.attention = PointerAttention(hidden_size)
    
    def encode(self, inputs):
        """
        Encode input sequence
        inputs: list of (input_size, 1) vectors
        """
        h = np.zeros((self.hidden_size, 1))
        encoder_states = []
        
        # 标准RNN递推:h_t = tanh(Wx*x_t + Wh*h_{t-1} + b),逐个吃进输入元素
        for x in inputs:
            h = np.tanh(
                np.dot(self.encoder_Wx, x) + 
                np.dot(self.encoder_Wh, h) + 
                self.encoder_b
            )
            encoder_states.append(h.flatten())
        
        # 返回所有时间步的隐状态(seq_len, hidden)供注意力查询,以及末状态h作为解码器初始状态
        return np.array(encoder_states), h
    
    def decode_step(self, x, h, encoder_states):
        """
        Single decoder step
        """
        # Update decoder hidden state
        h = np.tanh(
            np.dot(self.decoder_Wx, x) + 
            np.dot(self.decoder_Wh, h) + 
            self.decoder_b
        )
        
        # Compute pointer distribution
        # 用当前解码状态h对所有编码状态打分,得到(seq_len,1)的指针分布
        probs, scores = self.attention.forward(encoder_states, h)
        
        return probs, h, scores
    
    def forward(self, inputs, targets=None):
        """
        Full forward pass
        """
        # Encode inputs
        encoder_states, h = self.encode(inputs)
        
        # Decode (pointing to inputs)
        output_probs = []
        output_indices = []
        
        # Start token (use mean of inputs)
        # 用所有输入的均值当起始符,代替论文中专门的<start>向量
        x = np.mean([inp for inp in inputs], axis=0)
        
        for step in range(len(inputs)):
            probs, h, scores = self.decode_step(x, h, encoder_states)
            output_probs.append(probs)
            
            # Sample pointer
            # 贪心解码:取概率最大的输入位置作为本步的"指针"
            ptr_idx = np.argmax(probs)
            output_indices.append(ptr_idx)
            
            # Next input is the pointed element
            # 自回归:把被指向的输入元素喂给下一步解码器(类似seq2seq把上一步输出当输入)
            x = inputs[ptr_idx]
        
        return output_indices, output_probs

print("Pointer Network architecture created")

## Task: Convex Hull Problem(任务:凸包问题)

Given a set of 2D points, output them in convex hull order

给定一组二维点,按凸包(convex hull)顺序输出这些点

#### 💻 代码解读

**做什么:** 生成凸包任务的数据并画图展示:随机撒一些二维点,用现成算法算出凸包顶点的顺序,作为指针网络要学的"标准答案"。

**怎么做:**
- 定义 `generate_convex_hull_data` 函数:每个样本先用 `np.random.rand` 随机生成若干个 2D 点,再用 scipy 的 `ConvexHull` 算出凸包顶点的索引顺序 `hull_indices`(遇到算不出来的退化情况就跳过)。
- 把每个点转成列向量格式存进 `inputs`,连同原始点和答案一起打包成字典。
- 调用函数生成 10 个样本(每个 8 个点),打印实际生成的数量。
- 用 `plt.scatter` 画出第一个样本的所有点,再用红线依次连接凸包顶点(首尾相接围成一圈),并在每个点上标注编号。
- 最后打印凸包顶点顺序,比如 `[3, 7, 2, ...]`,意思是"按这个编号顺序走一圈,就能把所有点包在里面"。

In [ ]:
def generate_convex_hull_data(num_samples=20, num_points=10):
    """
    Generate random 2D points and their convex hull order
    """
    data = []
    
    for _ in range(num_samples):
        # Generate random points
        points = np.random.rand(num_points, 2)
        
        # Compute convex hull
        try:
            # 用scipy求凸包,hull.vertices给出按逆时针排列的顶点索引——这正是要学习的目标序列
            hull = ConvexHull(points)
            hull_indices = hull.vertices.tolist()
            
            # Convert points to input format
            # points[i:i+1]切片保持二维(1,2),转置成(2,1)列向量,符合网络输入格式
            inputs = [points[i:i+1].T for i in range(num_points)]
            
            data.append({
                'points': points,
                'inputs': inputs,
                'hull_indices': hull_indices
            })
        except:
            # Skip degenerate cases
            continue
    
    return data

# Generate data
convex_hull_data = generate_convex_hull_data(num_samples=10, num_points=8)
print(f"Generated {len(convex_hull_data)} convex hull examples")

# Visualize example
example = convex_hull_data[0]
points = example['points']
hull_indices = example['hull_indices']

plt.figure(figsize=(8, 8))
plt.scatter(points[:, 0], points[:, 1], s=100, alpha=0.6)

# Draw convex hull
for i in range(len(hull_indices)):
    start = hull_indices[i]
    # 取模让最后一个顶点连回第一个,闭合凸包多边形
    end = hull_indices[(i + 1) % len(hull_indices)]
    plt.plot([points[start, 0], points[end, 0]], 
             [points[start, 1], points[end, 1]], 
             'r-', linewidth=2)

# Label points
for i, (x, y) in enumerate(points):
    plt.text(x, y, str(i), fontsize=12, ha='center', va='center')

plt.title('Convex Hull Task')
plt.xlabel('X')
plt.ylabel('Y')
plt.grid(True, alpha=0.3)
plt.axis('equal')
plt.show()

print(f"\nConvex hull order: {hull_indices}")

## Test Pointer Network on Convex Hull(在凸包任务上测试指针网络)

#### 💻 代码解读

**做什么:** 用一个未训练的指针网络在凸包例子上跑一遍前向传播,并把每一步的注意力(指针)画出来,直观看到"网络在指谁"。

**怎么做:**
- 创建 `PointerNetwork(input_size=2, hidden_size=32)`(输入是二维坐标点)。
- 取第一个凸包样本,调用 `ptr_net.forward(inputs)` 得到预测的指向序列 `predicted_indices` 和每一步的概率 `probs`,并与真实凸包顺序 `true_hull` 对比打印(因为没训练,预测基本是乱指的)。
- 画一个 2x4 的子图网格,每个子图对应解码的一步:灰色圆点是所有输入点,红色圆圈的大小正比于该点获得的注意力权重 `attention_weights`——圈越大表示"手指越倾向指向它"。
- 每个子图的标题写明这一步最终指向了哪个点(`Step x: Point to y`)。
- 要点:这里展示的是"结构"而非"能力"——未训练时指得不准,但指针机制(输出=输入中的某个位置)已经完整运转。

In [ ]:
# Create pointer network
ptr_net = PointerNetwork(input_size=2, hidden_size=32)

# Test on example
test_example = convex_hull_data[0]
inputs = test_example['inputs']
true_hull = test_example['hull_indices']

# Forward pass (untrained)
# 权重是随机初始化的,所以预测基本是乱指——这里只是演示前向流程
predicted_indices, probs = ptr_net.forward(inputs)

print("Untrained Pointer Network:")
print(f"True convex hull order: {true_hull}")
print(f"Predicted order: {predicted_indices}")

# Visualize attention at each step
fig, axes = plt.subplots(2, 4, figsize=(16, 8))
# 把2x4的子图数组拉平成一维,方便用单个下标遍历
axes = axes.flatten()

for step in range(min(8, len(probs))):
    ax = axes[step]
    
    # Plot points
    ax.scatter(points[:, 0], points[:, 1], s=200, alpha=0.3, c='gray')
    
    # Highlight attention weights
    attention_weights = probs[step].flatten()
    for i, (x, y) in enumerate(points):
        # 用散点面积s正比于注意力权重,直观展示该步指针"看向"哪些点
        ax.scatter(x, y, s=1000*attention_weights[i], alpha=0.6, c='red')
        ax.text(x, y, str(i), fontsize=10, ha='center', va='center')
    
    ax.set_title(f'Step {step}: Point to {predicted_indices[step]}')
    ax.set_xlim(-0.1, 1.1)
    ax.set_ylim(-0.1, 1.1)
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.suptitle('Pointer Network Attention (Untrained)', y=1.02, fontsize=14)
plt.show()

## Simpler Task: Sort Numbers(更简单的任务:数字排序)

A simpler demonstration where the network learns to sort.

一个更简单的演示:让网络学习排序。

#### 💻 代码解读

**做什么:** 构造一个更简单的演示任务——数字排序:给一串随机数,正确答案是"按从小到大依次指向哪些位置",并画柱状图对比排序前后。

**怎么做:**
- 定义 `generate_sorting_data` 函数:每个样本用 `np.random.rand` 生成 `seq_len` 个随机数 `values`,再用 `np.argsort` 算出从小到大的索引顺序 `sorted_indices`(这就是指针网络该学的答案)。
- 把每个数包装成 1x1 的小矩阵存入 `inputs`,符合网络的输入格式。
- 生成 20 个长度为 6 的样本,取第一个打印:原始数值、排序后的索引顺序、以及按该顺序取出的排好序的数值。
- 用两张并排柱状图可视化:左图 `Original Order` 是原始顺序的数值,右图 `Sorted Order` 是按 `sorted_indices` 重排后的数值(柱子从矮到高)。
- 排序和凸包本质相同:输出都是"输入位置的一个排列",正是指针网络最擅长的题型。

In [ ]:
def generate_sorting_data(num_samples=50, seq_len=5):
    """
    Generate random sequences and their sorted order
    """
    data = []
    
    for _ in range(num_samples):
        # Random values
        values = np.random.rand(seq_len)
        
        # Sorted indices
        # argsort返回"按值从小到大排列的原始下标",正是指针网络要输出的目标序列
        sorted_indices = np.argsort(values).tolist()
        
        # Convert to input format (1D values)
        # 每个标量包成(1,1)列向量,统一成网络期望的输入形状
        inputs = [np.array([[v]]) for v in values]
        
        data.append({
            'values': values,
            'inputs': inputs,
            'sorted_indices': sorted_indices
        })
    
    return data

# Generate sorting data
sort_data = generate_sorting_data(num_samples=20, seq_len=6)

# Test example
example = sort_data[0]
print("Sorting Task Example:")
print(f"Values: {example['values']}")
print(f"Sorted order (indices): {example['sorted_indices']}")
print(f"Sorted values: {example['values'][example['sorted_indices']]}")

# Visualize
plt.figure(figsize=(12, 4))
plt.subplot(1, 2, 1)
plt.bar(range(len(example['values'])), example['values'])
plt.title('Original Order')
plt.xlabel('Index')
plt.ylabel('Value')

plt.subplot(1, 2, 2)
# 花式索引:用索引列表一次性按排序顺序取出所有值
sorted_vals = example['values'][example['sorted_indices']]
plt.bar(range(len(sorted_vals)), sorted_vals)
plt.title('Sorted Order')
plt.xlabel('Position in Sorted Sequence')
plt.ylabel('Value')

plt.tight_layout()
plt.show()

## Key Takeaways(核心要点)

### Pointer Networks Innovation:
1. **Output vocabulary is the input**: Network points to input elements
2. **Variable output size**: Can handle different input lengths
3. **No fixed vocabulary**: Solves combinatorial problems
4. **Attention as selection**: Uses attention mechanism to "point"

### Applications:
- Convex hull computation
- Traveling salesman problem (TSP)
- Sorting
- Delaunay triangulation
- Any problem where output is a permutation/subset of input

### Architecture Components:
1. **Encoder**: Processes input sequence
2. **Decoder**: Generates sequence of pointers
3. **Attention**: Computes distribution over input positions
4. **Pointing**: Selects input element to output next

### Training:
- Supervised learning with correct pointer sequences
- Cross-entropy loss on pointer distributions
- Can use reinforcement learning for optimization problems

### 指针网络的创新之处:
1. **输出词表就是输入**:网络直接指向输入元素
2. **可变的输出规模**:能够处理不同长度的输入
3. **无需固定词表**:可解决组合优化问题
4. **注意力即选择**:利用注意力(attention)机制来"指向"

### 应用场景:
- 凸包(convex hull)计算
- 旅行商问题(TSP)
- 排序
- Delaunay 三角剖分
- 任何输出是输入的排列/子集的问题

### 架构组成:
1. **编码器(Encoder)**:处理输入序列
2. **解码器(Decoder)**:生成指针序列
3. **注意力(Attention)**:计算输入位置上的概率分布
4. **指向(Pointing)**:选择下一个要输出的输入元素

### 训练方式:
- 使用正确指针序列进行监督学习
- 对指针分布使用交叉熵(cross-entropy)损失
- 对优化问题可采用强化学习(reinforcement learning)